# TMJ Binary Position Classifier — Detector-Based Crops

Ноутбук для обучения **бинарного классификатора** положения ВНЧС (центральное / нецентральное).

**Approach A**: NIfTI-кропы 128³ вокруг предсказаний детектора (вместо центральных кропов).  
**Approach B**: `BinaryFocalLoss` (или `BCEWithLogitsLoss` + `pos_weight`) + калибровка порога через Youden's J.

**Стабилизация обучения (план диагностики):** §4.0 — распределение классов и leakage; §6.0 — sanity-check на 32 сэмплах; AdamW + warmup + cosine LR; gradient clipping; накопление градиента (`BATCH_SIZE` × `GRAD_ACCUM_STEPS`); чекпоинт по **val AUC** (при `nan` — по accuracy).

Поддерживаемые среды: **Yandex DataSphere** (основная) · **Google Colab** · **Local**

In [ ]:
%pip install -q --upgrade scipy tqdm nibabel scikit-learn pydicom pylibjpeg pylibjpeg-libjpeg

import subprocess, sys
from pathlib import Path

FS = Path("/home/jupyter/filestore")  # filestore — единственное writable хранилище

LEFT_DETECTOR_URL  = "https://github.com/tzopiz/MasterProject/releases/download/heatmap-detector-v1/left_detector.pth"
RIGHT_DETECTOR_URL = "https://github.com/tzopiz/MasterProject/releases/download/heatmap-detector-v1/right_detector.pth"
LABELS_URL         = "https://github.com/tzopiz/MasterProject/releases/download/crops-v1/tmj_position_labels.json"
MANIFEST_URL       = "https://github.com/tzopiz/MasterProject/releases/download/crops-v1/manifest.json"

def wget(url, dest):
    dest = Path(dest)
    if dest.exists():
        print(f"  уже есть: {dest} ({dest.stat().st_size/1e6:.1f} MB)")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  скачиваю {dest.name}...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(dest), url], check=True)
    print(f"  готово: {dest.stat().st_size/1e6:.1f} MB")

if Path("/home/jupyter").exists():  # DataSphere
    wget(LEFT_DETECTOR_URL,  FS / "models" / "left_detector.pth")
    wget(RIGHT_DETECTOR_URL, FS / "models" / "right_detector.pth")
    wget(LABELS_URL,         FS / "tmj_position_labels.json")
    wget(MANIFEST_URL,       FS / "manifest.json")

In [ ]:
import os, sys
from pathlib import Path

IN_DATASPHERE = Path("/home/jupyter").exists()
IN_COLAB = "google.colab" in sys.modules
env_name = "DataSphere" if IN_DATASPHERE else ("Colab" if IN_COLAB else "Local")
print(f"Среда: {env_name}")

if IN_DATASPHERE:
    FS                  = Path("/home/jupyter/filestore")
    MANIFEST_PATH       = FS / "manifest.json"
    LABELS_PATH         = FS / "tmj_position_labels.json"
    LEFT_DETECTOR_PATH  = FS / "models" / "left_detector.pth"
    RIGHT_DETECTOR_PATH = FS / "models" / "right_detector.pth"
    CROPS_DIR           = FS / "detector_crops_v2"
    OUTPUT_DIR          = FS / "experiments"
    DATASET_ROOT        = None  # не используется: кропы скачиваются готовые

elif IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DATA_ROOT           = Path("/content/drive/MyDrive/tmj_data")
    MANIFEST_PATH       = DATA_ROOT / "dataset_public" / "manifest_private.json"
    LABELS_PATH         = DATA_ROOT / "tmj_position_labels.json"
    LEFT_DETECTOR_PATH  = DATA_ROOT / "models" / "left_detector.pth"
    RIGHT_DETECTOR_PATH = DATA_ROOT / "models" / "right_detector.pth"
    CROPS_DIR           = Path("/content/detector_crops_v2")
    OUTPUT_DIR          = Path("/content/experiments")
    DATASET_ROOT        = DATA_ROOT / "dataset_public"

else:  # Local
    ROOT                = Path(".").resolve().parent
    MANIFEST_PATH       = ROOT / "data" / "dataset_cbct_public" / "manifest_private.json"
    LABELS_PATH         = ROOT / "data" / "tmj_position_labels.json"
    LEFT_DETECTOR_PATH  = ROOT / "models" / "checkpoints" / "left_detector.pth"
    RIGHT_DETECTOR_PATH = ROOT / "models" / "checkpoints" / "right_detector.pth"
    CROPS_DIR           = ROOT / "data" / "detector_crops_v2"
    OUTPUT_DIR          = ROOT / "experiments"
    DATASET_ROOT        = ROOT / "data" / "dataset_cbct_public"

CROPS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MANIFEST_PATH       : {MANIFEST_PATH}  exists={MANIFEST_PATH.exists()}")
print(f"LABELS_PATH         : {LABELS_PATH}  exists={LABELS_PATH.exists()}")
print(f"LEFT_DETECTOR_PATH  : {LEFT_DETECTOR_PATH}  exists={LEFT_DETECTOR_PATH.exists()}")
print(f"RIGHT_DETECTOR_PATH : {RIGHT_DETECTOR_PATH}  exists={RIGHT_DETECTOR_PATH.exists()}")
print(f"CROPS_DIR           : {CROPS_DIR}")

In [3]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda"); print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps"); print("MPS")
else:
    device = torch.device("cpu"); print("CPU")
print(f"PyTorch: {torch.__version__}")

GPU: Tesla V100-PCIE-32GB
PyTorch: 2.0.1+cu118


## 2. Label Table

In [ ]:
import json, random, logging
from pathlib import Path
from typing import Dict, List, Tuple

logger = logging.getLogger("tmj")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")


def map_sagittal(code):
    if code not in (1, 2, 3): raise ValueError(f"Invalid sagittal code: {code}")
    return code - 1


def map_frontal(code):
    if code not in (4, 5, 6): raise ValueError(f"Invalid frontal code: {code}")
    return code - 4


def build_index(manifest_path, labels_path, dataset_root, cache_path=None):
    with open(manifest_path, "r", encoding="utf-8") as f: manifest = json.load(f)
    with open(labels_path, "r", encoding="utf-8") as f: labels_data = json.load(f)
    label_by_name = {p["name_raw"].strip(): p["labels"] for p in labels_data["patients"]}
    records, skipped = [], 0
    for study in manifest["studies"]:
        name = study["patient_name"].strip()
        if name not in label_by_name: skipped += 1; continue
        lbl = label_by_name[name]
        records.append({
            "study_id": study["study_id"],
            "dicom_dir": str(Path(dataset_root) / study["study_id"]),
            "patient_name": name,
            "sag_right": map_sagittal(lbl["sagittal"]["right"]),
            "sag_left":  map_sagittal(lbl["sagittal"]["left"]),
            "fr_right":  map_frontal(lbl["frontal"]["right"]),
            "fr_left":   map_frontal(lbl["frontal"]["left"]),
        })
    logger.info("build_index: %d matched, %d skipped", len(records), skipped)
    return records


def binarize_labels(records, crop_dir):
    crop_dir = Path(crop_dir).resolve()
    out = []
    for rec in records:
        for side in ("left", "right"):
            out.append({
                "study_id": rec["study_id"],
                "patient_name": rec["patient_name"],
                "side": side,
                "sag": 0 if rec[f"sag_{side}"] == 0 else 1,
                "fr":  0 if rec[f"fr_{side}"]  == 0 else 1,
                "crop_path": str(crop_dir / rec["study_id"] / f"{rec['study_id']}_{side}.nii.gz"),
            })
    logger.info("binarize_labels: %d → %d binary records", len(records), len(out))
    return out


def split_by_patient(records, split_ratio=0.8, seed=42):
    patients = sorted(set(r["patient_name"] for r in records))
    rng = random.Random(seed); rng.shuffle(patients)
    n = len(patients)
    idx = min(max(1, int(n * split_ratio)), n - 1)
    train_p = set(patients[:idx]); val_p = set(patients[idx:])
    train = [r for r in records if r["patient_name"] in train_p]
    val   = [r for r in records if r["patient_name"] in val_p]
    logger.info("split: train=%d val=%d", len(train), len(val))
    return train, val


def split_by_patient_stratified(records, split_ratio=0.8, seed=42):
    """
    Split by patient ensuring both sag classes appear in val.
    Patients sorted by sag-positive fraction, then alternately assigned to train/val.
    """
    from collections import defaultdict
    patient_records = defaultdict(list)
    for r in records:
        patient_records[r["patient_name"]].append(r)

    # Compute sag=1 fraction per patient
    patients_info = []
    for name, recs in patient_records.items():
        frac = sum(r["sag"] for r in recs) / len(recs)
        patients_info.append((name, frac))

    # Sort by fraction, then interleave into train/val
    patients_info.sort(key=lambda x: x[1])
    rng = random.Random(seed)

    # Assign every Nth patient to val to spread class distribution
    n_val = max(1, int(len(patients_info) * (1 - split_ratio)))
    # Take evenly spaced indices for val
    step = len(patients_info) / n_val
    val_idx = set(int(i * step) for i in range(n_val))

    train_names = {p[0] for i, p in enumerate(patients_info) if i not in val_idx}
    val_names   = {p[0] for i, p in enumerate(patients_info) if i in val_idx}

    train = [r for r in records if r["patient_name"] in train_names]
    val   = [r for r in records if r["patient_name"] in val_names]

    # Log class distribution in each split
    for split_name, split_recs in [("Train", train), ("Val", val)]:
        n0 = sum(1 for r in split_recs if r["sag"] == 0)
        n1 = sum(1 for r in split_recs if r["sag"] == 1)
        logger.info("%s sag: %d central (%.0f%%) / %d non-central (%.0f%%)",
                    split_name, n0, 100*n0/max(len(split_recs),1),
                    n1, 100*n1/max(len(split_recs),1))
    return train, val


# DATASET_ROOT=None в DataSphere (DICOM не нужен — кропы уже готовы)
_dset_root = str(DATASET_ROOT) if DATASET_ROOT else str(CROPS_DIR)
all_records = build_index(str(MANIFEST_PATH), str(LABELS_PATH), _dset_root)
binary_records = binarize_labels(all_records, crop_dir=str(CROPS_DIR))

from collections import Counter
sag_dist = Counter(r["sag"] for r in binary_records)
fr_dist  = Counter(r["fr"]  for r in binary_records)
print(f"Записей: {len(all_records)} исследований → {len(binary_records)} кропов")
print(f"Sagittal: central={sag_dist[0]} non-central={sag_dist[1]}")
print(f"Frontal:  central={fr_dist[0]}  non-central={fr_dist[1]}")

In [ ]:
SPLIT_RATIO = 0.8
train_records, val_records = split_by_patient_stratified(binary_records, split_ratio=SPLIT_RATIO, seed=42)
print(f"Train: {len(train_records)}  Val: {len(val_records)}")

## 3. Preprocessing: Detector → Crops (один раз)

In [ ]:
existing = list(CROPS_DIR.rglob("*.nii.gz"))
print(f"Кропов уже есть: {len(existing)}, ожидается: {len(all_records)*2}")

if len(existing) < len(all_records) * 2:
    if IN_DATASPHERE:
        # DataSphere: скачиваем готовые кропы (сгенерированы heatmap-детектором v1)
        import subprocess, tarfile
        CROPS_URL = "https://github.com/tzopiz/MasterProject/releases/download/crops-v2/detector_crops_v2.tar.gz"
        tar_path = CROPS_DIR.parent / "detector_crops_v2.tar.gz"
        print("Скачиваю кропы crops-v2 (~800 MB)...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(tar_path), CROPS_URL], check=True)
        print("Распаковываю...")
        with tarfile.open(tar_path) as tf:
            tf.extractall(CROPS_DIR.parent)
        tar_path.unlink()
        n_crops = len(list(CROPS_DIR.rglob("*.nii.gz")))
        print(f"Готово: {n_crops} кропов")
    else:
        # Локально / Colab: генерируем из DICOM с heatmap-детекторами
        import torch.nn as nn
        import torch.nn.functional as F
        from scipy import ndimage
        import nibabel as nib
        from tqdm.notebook import tqdm

        TARGET_SHAPE = (96, 128, 128)

        # ── Inline model definition ──
        def _dconv(a, b):
            return nn.Sequential(
                nn.Conv3d(a,b,3,padding=1,bias=False), nn.BatchNorm3d(b), nn.ReLU(inplace=True),
                nn.Conv3d(b,b,3,padding=1,bias=False), nn.BatchNorm3d(b), nn.ReLU(inplace=True),
            )
        class _Enc(nn.Module):
            def __init__(self,a,b): super().__init__(); self.c=_dconv(a,b); self.p=nn.MaxPool3d(2)
            def forward(self,x): s=self.c(x); return self.p(s),s
        class _Dec(nn.Module):
            def __init__(self,a,b,c):
                super().__init__()
                self.u=nn.ConvTranspose3d(a,a//2,2,stride=2); self.c=_dconv(a//2+b,c)
            def forward(self,x,s):
                x=self.u(x)
                if x.shape!=s.shape: x=F.pad(x,[0,s.shape[4]-x.shape[4],0,s.shape[3]-x.shape[3],0,s.shape[2]-x.shape[2]])
                return self.c(torch.cat([s,x],1))
        class TMJHeatmapDetector(nn.Module):
            def __init__(self,feats=None):
                super().__init__()
                feats=feats or [32,64,128,256]
                self.encs=nn.ModuleList(); prev=1
                for f in feats: self.encs.append(_Enc(prev,f)); prev=f
                self.bot=_dconv(feats[-1],feats[-1]*2); prev=feats[-1]*2
                self.decs=nn.ModuleList()
                for f in reversed(feats): self.decs.append(_Dec(prev,f,f)); prev=f
                self.head=nn.Conv3d(feats[0],1,1)
            def forward(self,x):
                skips=[]
                for e in self.encs: x,s=e(x); skips.append(s)
                x=self.bot(x)
                for d,s in zip(self.decs,reversed(skips)): x=d(x,s)
                return self.head(x)

        def load_heatmap_model(path):
            ck=torch.load(path, map_location="cpu")
            m=TMJHeatmapDetector(); m.load_state_dict(ck["model_state_dict"]); m.eval().to(device)
            print(f"  {Path(path).name}: ep={ck.get('epoch','?')} MAE={ck.get('best_val_mae',float('nan')):.2f}ds")
            return m

        print("Загружаю детекторы...")
        left_det  = load_heatmap_model(str(LEFT_DETECTOR_PATH))
        right_det = load_heatmap_model(str(RIGHT_DETECTOR_PATH))

        def prep_volume(vol):
            p2,p98 = np.percentile(vol,[2,98])
            v = np.clip(vol,p2,p98); v=(v-p2)/max(p98-p2,1e-6)
            orig = np.array(v.shape,dtype=float)
            if tuple(v.shape)!=TARGET_SHAPE:
                z=[t/s for t,s in zip(TARGET_SHAPE,v.shape)]
                v=ndimage.zoom(v.astype(np.float32),z,order=1)
            return torch.tensor(v,dtype=torch.float32).unsqueeze(0).unsqueeze(0), orig

        def argmax_orig(hm, orig_shape):
            idx=np.unravel_index(hm.argmax(),hm.shape)
            c=np.array(idx,dtype=float)
            sc=orig_shape/np.array(TARGET_SHAPE,dtype=float)
            return np.clip((c*sc).astype(int),0,orig_shape.astype(int)-1)

        def extract_crop(vol, center, sz=128):
            D,H,W=vol.shape; h=sz//2
            z,y,x=int(center[0]),int(center[1]),int(center[2])
            crop=vol[max(0,z-h):min(D,z+h), max(0,y-h):min(H,y+h), max(0,x-h):min(W,x+h)]
            if crop.shape!=(sz,sz,sz):
                pad=np.zeros((sz,sz,sz),dtype=crop.dtype)
                pz=(sz-crop.shape[0])//2; py=(sz-crop.shape[1])//2; px_=(sz-crop.shape[2])//2
                pad[pz:pz+crop.shape[0],py:py+crop.shape[1],px_:px_+crop.shape[2]]=crop
                crop=pad
            return crop

        for rec in tqdm(all_records, desc="Generating crops v2"):
            sid = rec["study_id"]
            out_dir = CROPS_DIR / sid
            if (out_dir / f"{sid}_left.nii.gz").exists() and (out_dir / f"{sid}_right.nii.gz").exists():
                continue
            out_dir.mkdir(parents=True, exist_ok=True)
            raw = load_dicom_volume(rec["dicom_dir"])
            inp, orig = prep_volume(raw)
            inp = inp.to(device)
            with torch.no_grad():
                lh = torch.sigmoid(left_det(inp)).squeeze().cpu().numpy()
                rh = torch.sigmoid(right_det(inp)).squeeze().cpu().numpy()
            lc = argmax_orig(lh, orig)
            rc = argmax_orig(rh, orig)
            nib.save(nib.Nifti1Image(extract_crop(raw, lc), np.eye(4)), str(out_dir/f"{sid}_left.nii.gz"))
            nib.save(nib.Nifti1Image(extract_crop(raw, rc), np.eye(4)), str(out_dir/f"{sid}_right.nii.gz"))
        print("Готово!")
else:
    print("Кропы уже есть — пропускаем.")

In [ ]:
missing = [r for r in binary_records if not Path(r["crop_path"]).exists()]
print(f"Кропов не найдено: {len(missing)} из {len(binary_records)}")
if missing:
    print("Примеры:", [r["crop_path"] for r in missing[:3]])

## 4. Dataset & DataLoader

In [ ]:
import random as _random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
from tqdm.notebook import tqdm


def augment_volume(vol: np.ndarray) -> np.ndarray:
    """
    Консервативная аугментация для малого датасета (142 сэмпла).
    Только операции, которые не искажают анатомию.
    """
    # Random axis flips — анатомически безопасно
    for ax in range(3):
        if _random.random() < 0.5:
            vol = np.flip(vol, axis=ax).copy()

    # Лёгкий intensity jitter (контраст ±5%, яркость ±3%)
    # Симулирует вариацию протокола/аппарата сканирования
    if _random.random() < 0.7:
        alpha = _random.uniform(0.95, 1.05)
        beta  = _random.uniform(-0.03, 0.03)
        vol   = np.clip(alpha * vol + beta, 0.0, 1.0)

    # Лёгкий Gaussian noise (sigma=0.01) — регуляризация
    if _random.random() < 0.5:
        vol = np.clip(vol + np.random.normal(0, 0.01, vol.shape).astype(np.float32), 0.0, 1.0)

    return vol.astype(np.float32)


class TMJBinaryPositionDataset(Dataset):
    def __init__(self, records, is_train=False):
        self.records  = records
        self.is_train = is_train
        print(f"Кеширую {len(records)} томов в RAM...")
        self.cache = []
        for rec in tqdm(records, leave=False):
            img = nib.load(rec["crop_path"])
            vol = np.asarray(img.dataobj, dtype=np.float32)
            p2, p98 = np.percentile(vol, [2, 98])
            vol = np.clip(vol, p2, p98)
            denom = p98 - p2
            vol = (vol - p2) / denom if denom > 0 else np.zeros_like(vol)
            self.cache.append(vol)
        ram_gb = sum(v.nbytes for v in self.cache) / 1e9
        print(f"Кеш готов: {len(self.cache)} томов, {ram_gb:.2f} GB RAM")

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        vol = self.cache[idx].copy()
        if self.is_train:
            vol = augment_volume(vol)
        return (
            torch.from_numpy(vol).float().unsqueeze(0),
            torch.tensor([self.records[idx]["sag"], self.records[idx]["fr"]], dtype=torch.long)
        )


# Эффективный batch ≈ BATCH_SIZE * GRAD_ACCUM_STEPS (план: 32–64 или накопление градиента)
BATCH_SIZE       = 16
GRAD_ACCUM_STEPS = 2
NUM_WORKERS      = 0

train_ds = TMJBinaryPositionDataset(train_records, is_train=True)
val_ds   = TMJBinaryPositionDataset(val_records,   is_train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)

print(f"Train: {len(train_ds)} сэмплов, {len(train_loader)} батчей")
print(f"Val:   {len(val_ds)} сэмплов, {len(val_loader)} батчей")

vol, lbl = next(iter(train_loader))
print(f"volume: {vol.shape} [{vol.min():.2f}, {vol.max():.2f}]  labels: {lbl[:4]}")


### 4.0 Диагностика данных (шаг 0 плана)

Распределение классов, соотношение, пересечение train/val, случайные записи, статистика пикселей после нормализации и один батч из `val_loader` (без аугментации). Запускайте **после** создания `train_records`, `val_records`, `train_loader`, `val_loader` и **до** полного обучения.

In [ ]:
from collections import Counter
from pathlib import Path

print("=== Распределение классов (sagittal) ===")
tr_sag = [r["sag"] for r in train_records]
va_sag = [r["sag"] for r in val_records]
ctr_tr = Counter(tr_sag)
ctr_va = Counter(va_sag)
n0_tr, n1_tr = ctr_tr.get(0, 0), ctr_tr.get(1, 0)
n0_va, n1_va = ctr_va.get(0, 0), ctr_va.get(1, 0)
print(f"Train: 0={n0_tr}  1={n1_tr}  |  ratio max/min = {max(n0_tr, n1_tr) / max(min(n0_tr, n1_tr), 1):.2f}")
print(f"Val:   0={n0_va}  1={n1_va}  |  ratio max/min = {max(n0_va, n1_va) / max(min(n0_va, n1_va), 1):.2f}")
if max(n0_tr, n1_tr) / max(min(n0_tr, n1_tr), 1) > 3:
    print("⚠ Train: дисбаланс > 1:3 — критично; используйте pos_weight / focal / WeightedRandomSampler.")

print("\n=== Пересечение train/val (leakage) ===")
tr_p = {r["patient_name"] for r in train_records}
va_p = {r["patient_name"] for r in val_records}
tr_paths = {r["crop_path"] for r in train_records}
va_paths = {r["crop_path"] for r in val_records}
print(f"Общих пациентов: {len(tr_p & va_p)} (ожидается 0)")
print(f"Общих crop_path: {len(tr_paths & va_paths)} (ожидается 0)")

print("\n=== 15 случайных train-записей (метки) ===")
_rng_d = np.random.default_rng(42)
ix = _rng_d.choice(len(train_records), size=min(15, len(train_records)), replace=False)
for i in ix:
    rec = train_records[int(i)]
    tail = Path(rec["crop_path"]).name
    print(f"  {rec['study_id'][:14]}  {rec['side']:5}  sag={rec['sag']}  fr={rec['fr']}  |  {tail}")

print("\n=== Кеш val: min / max / mean / std (первые до 8 томов) ===")
if len(val_ds) > 0:
    sample = [val_ds.cache[i] for i in range(min(8, len(val_ds)))]
    stack = np.concatenate([v.ravel() for v in sample])
    print(f"  min={stack.min():.4f}  max={stack.max():.4f}  mean={stack.mean():.4f}  std={stack.std():.4f}  (ожидается [0,1] после p2–p98)")

print("\n=== Один батч val_loader ===")
_vb, _lb = next(iter(val_loader))
print(f"  volume: {_vb.shape}  dtype={_vb.dtype}  range=[{_vb.min():.4f}, {_vb.max():.4f}]")
print(f"  labels: {_lb.shape}  dtype={_lb.dtype}  unique={torch.unique(_lb).tolist()}  (sag в [:,0], frontal в [:,1])")

### 4.1 Примеры объёмов (train / val)

Случайные кропы из кеша датасета (та же нормализация, **без** train-аугментации). Плоскости: середина по Z / Y / X.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def _triplanar(vol_np, axes3, cmap="gray"):
    D, H, W = vol_np.shape
    axes3[0].imshow(vol_np[D // 2, :, :], cmap=cmap, vmin=0, vmax=1)
    axes3[0].set_title("mid-Z"); axes3[0].axis("off")
    axes3[1].imshow(vol_np[:, H // 2, :], cmap=cmap, vmin=0, vmax=1)
    axes3[1].set_title("mid-Y"); axes3[1].axis("off")
    axes3[2].imshow(vol_np[:, :, W // 2], cmap=cmap, vmin=0, vmax=1)
    axes3[2].set_title("mid-X"); axes3[2].axis("off")


def _class_label_binary(y: int) -> str:
    return "центральное" if int(y) == 0 else "нецентральное"


def _sag_fr_caption(rec):
    sg, fg = int(rec["sag"]), int(rec["fr"])
    return (
        f"В датасете — sagittal: класс {sg} ({_class_label_binary(sg)})  ·  "
        f"frontal: класс {fg} ({_class_label_binary(fg)})"
    )


_rng = np.random.default_rng(42)
N_TR, N_VA = 4, 4
tr_ix = _rng.choice(len(train_ds), size=min(N_TR, len(train_ds)), replace=False)
va_ix = _rng.choice(len(val_ds), size=min(N_VA, len(val_ds)), replace=False)
nrows = len(tr_ix) + len(va_ix)
fig, axes = plt.subplots(nrows, 3, figsize=(10, 2.6 * nrows))
if nrows == 1:
    axes = np.asarray([axes])
fig.suptitle(
    "Датасет: примеры кропов  ·  классы разметки: 0 = центральное, 1 = нецентральное (sagittal / frontal)",
    fontsize=11,
    fontweight="bold",
)

r = 0
for idx in tr_ix:
    idx = int(idx)
    vol = train_ds.cache[idx]
    rec = train_ds.records[idx]
    _triplanar(vol, axes[r])
    axes[r, 0].set_ylabel(
        f"train\n{rec['side']}\n{rec['study_id'][:14]}",
        fontsize=8,
        rotation=0,
        labelpad=36,
        va="center",
    )
    axes[r, 1].set_title(_sag_fr_caption(rec), fontsize=9)
    r += 1

for idx in va_ix:
    idx = int(idx)
    vol = val_ds.cache[idx]
    rec = val_ds.records[idx]
    _triplanar(vol, axes[r])
    axes[r, 0].set_ylabel(
        f"val\n{rec['side']}\n{rec['study_id'][:14]}",
        fontsize=8,
        rotation=0,
        labelpad=36,
        va="center",
    )
    axes[r, 1].set_title(_sag_fr_caption(rec), fontsize=9)
    r += 1

plt.tight_layout()
plt.show()


## 5. Model

In [ ]:
import torch.nn as nn
from typing import List, Optional

def _conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        nn.MaxPool3d(2, 2),
    )

class TMJSagittalClassifier(nn.Module):
    """Single-head binary classifier for sagittal TMJ position (central vs non-central)."""
    def __init__(self, in_channels=1, features=None, fc_hidden=256, dropout=0.5):
        super().__init__()
        if features is None: features = [16, 32, 64, 128]
        blocks, prev = [], in_channels
        for oc in features:
            blocks.append(_conv_block(prev, oc)); prev = oc
        self.backbone    = nn.Sequential(*blocks)
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Linear(prev, fc_hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(fc_hidden, 1)
        )
    def forward(self, x):
        f = self.backbone(x)
        f = self.global_pool(f).view(f.size(0), -1)
        return self.head(f)  # (B, 1) logit

model = TMJSagittalClassifier().to(device)
n = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n/1e6:.2f}M")
with torch.no_grad():
    logit = model(torch.randn(1,1,32,48,48).to(device))
print(f"Output: {logit.shape}")

## 6. Training

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        if reduction not in ("mean", "sum", "none"):
            raise ValueError(f"Bad reduction: {reduction}")
        self.gamma, self.alpha, self.reduction = gamma, alpha, reduction

    def forward(self, logits, targets):
        if logits.dim() == 2 and logits.shape[1] == 1:
            logits = logits.squeeze(1)
        targets = targets.float().to(logits.device)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = (1 - p_t).pow(self.gamma) * bce
        if self.alpha is not None:
            loss = (self.alpha * targets + (1 - self.alpha) * (1 - targets)) * loss
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


# --- Гиперпараметры (план: ниже LR, warmup+cosine, ранняя остановка по AUC, дисбаланс) ---
EPOCHS = 150
LR = 3e-5
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
LR_MIN = 1e-7
EARLY_STOPPING = 25
GAMMA = 2.0
GRAD_CLIP_NORM = 1.0

# "focal" — как раньше; "bce" — BCEWithLogitsLoss + pos_weight (n_neg / n_pos)
LOSS_TYPE = "focal"

# Распределение классов (sagittal)
n0 = sum(1 for r in train_records if r["sag"] == 0)
n1 = sum(1 for r in train_records if r["sag"] == 1)
total = n0 + n1
alpha_sag = n0 / total
print(f"Train sag: 0={n0} ({100 * n0 / total:.1f}%)  1={n1} ({100 * n1 / total:.1f}%)  focal α={alpha_sag:.3f}")

n0v = sum(1 for r in val_records if r["sag"] == 0)
n1v = sum(1 for r in val_records if r["sag"] == 1)
print(f"Val   sag: 0={n0v} ({100 * n0v / max(n0v + n1v, 1):.1f}%)  1={n1v} ({100 * n1v / max(n0v + n1v, 1):.1f}%)")

_cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.array([r["sag"] for r in train_records]))
print(f"sklearn balanced class_weight [0,1]: {_cw[0]:.4f}, {_cw[1]:.4f}")
pos_weight = torch.tensor([float(n0) / float(max(n1, 1))], dtype=torch.float32)
print(f"BCE pos_weight (n0/n1): {pos_weight.item():.4f}")

# --- Шаг 3 плана: голова выдаёт logits, loss — по logits ---
model.eval()
with torch.no_grad():
    _xb, _yb = next(iter(val_loader))
    _lo = model(_xb.to(device))
print(
    f"Проверка выхода модели: shape={_lo.shape}  logits in [{_lo.min().item():.3f}, {_lo.max().item():.3f}]  "
    f"(ожидаются сырые logits; без Sigmoid в конце TMJSagittalClassifier)"
)


### 6.0 Sanity-check: переобучение на малой подвыборке (шаг 7 плана)

Запускайте **после** ячейки с `BinaryFocalLoss` и гиперпараметрами, **до** ячейки с `optimizer`.
Если за ~50–100 шагов loss не падает почти к нулю на 1–2 батчах — проверьте данные, метки и что голова выдаёт **logits** (без двойного sigmoid).


In [ ]:
from torch.utils.data import DataLoader, Subset

SANITY_SAMPLES = min(32, len(train_ds))
if SANITY_SAMPLES < 4:
    print("Слишком мало данных для sanity-check.")
else:
    _ix = list(range(SANITY_SAMPLES))
    _sub = Subset(train_ds, _ix)
    _ld = DataLoader(_sub, batch_size=min(16, SANITY_SAMPLES), shuffle=True, num_workers=0)
    _m = TMJSagittalClassifier().to(device)
    _opt = torch.optim.AdamW(_m.parameters(), lr=1e-3, weight_decay=0.0)
    _crit = (
        BinaryFocalLoss(gamma=GAMMA, alpha=alpha_sag)
        if LOSS_TYPE == "focal"
        else nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    )
    _m.train()
    for ep in range(1, 81):
        tot = 0.0
        for vols, labels in _ld:
            vols = vols.to(device)
            y = labels[:, 0].float().to(device)
            _opt.zero_grad()
            logits = _m(vols)
            loss = _crit(logits, y)
            loss.backward()
            _opt.step()
            tot += loss.item()
        if ep % 20 == 0 or ep == 1:
            print(f"  sanity ep {ep:3d}  mean_loss={tot / max(len(_ld),1):.5f}")
    del _m, _opt, _ld, _sub
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("Sanity-check завершён (отдельная модель, основное обучение не затронуто).")


In [ ]:
import datetime
import json as _json2
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

if LOSS_TYPE == "focal":
    criterion = BinaryFocalLoss(gamma=GAMMA, alpha=alpha_sag)
else:
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

_warm = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
_cos_T = max(1, EPOCHS - WARMUP_EPOCHS)
_cos = CosineAnnealingLR(optimizer, T_max=_cos_T, eta_min=LR_MIN)
scheduler = SequentialLR(optimizer, schedulers=[_warm, _cos], milestones=[WARMUP_EPOCHS])

scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
print(f"Mixed precision: {'ON' if scaler else 'OFF'}")
print(f"Optimizer: AdamW  lr={LR}  scheduler=warmup({WARMUP_EPOCHS})+cosine(T_max={_cos_T})  loss={LOSS_TYPE}")

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
exp_dir = OUTPUT_DIR / f"sag_only_{timestamp}"
exp_dir.mkdir(parents=True, exist_ok=True)

config = {
    "task": "sagittal_only_binary",
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "lr_min": LR_MIN,
    "loss_type": LOSS_TYPE,
    "gamma": GAMMA,
    "alpha_sag": alpha_sag,
    "pos_weight_n0_over_n1": float(pos_weight.item()),
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "early_stopping_patience": EARLY_STOPPING,
    "early_stopping_monitor": "val_auc",
    "split_ratio": SPLIT_RATIO,
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_sag_0": n0,
    "train_sag_1": n1,
    "val_sag_0": n0v,
    "val_sag_1": n1v,
}
with open(exp_dir / "config.json", "w") as f:
    _json2.dump(config, f, indent=2)
print(f"Эксперимент: {exp_dir}")


In [ ]:
from tqdm.notebook import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score


def compute_detailed_metrics(logits, labels, thresh=0.5):
    """Accuracy, sensitivity (recall class 1), specificity (recall class 0), F1."""
    probs = torch.sigmoid(logits.squeeze(1))
    preds = (probs >= thresh).long()
    labels = labels.long()

    tp = ((preds == 1) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * sens / max(prec + sens, 1e-8)
    return {"acc": acc, "sensitivity": sens, "specificity": spec, "f1": f1, "tp": tp, "tn": tn, "fp": fp, "fn": fn}


def run_epoch(train_mode, epoch):
    model.train() if train_mode else model.eval()
    loader = train_loader if train_mode else val_loader
    tag = "Train" if train_mode else "Val  "

    rl = 0.0
    all_logits, all_labels_list = [], []

    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx:
        if train_mode:
            optimizer.zero_grad()
        for i, (vols, labels) in enumerate(tqdm(loader, desc=f"[{epoch}] {tag}", leave=False)):
            vols, labels = vols.to(device), labels.to(device)
            sag_labels = labels[:, 0].float()
            n_accum = GRAD_ACCUM_STEPS if train_mode else 1
            is_last = (i + 1) == len(loader)
            rem = (i + 1) % n_accum
            if train_mode and is_last and rem != 0:
                loss_denom = rem
            else:
                loss_denom = n_accum
            is_step = ((i + 1) % n_accum == 0) or is_last

            if train_mode:
                if scaler:
                    with torch.cuda.amp.autocast():
                        logit = model(vols)
                        loss = criterion(logit, sag_labels) / loss_denom
                    scaler.scale(loss).backward()
                    if is_step:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()
                else:
                    logit = model(vols)
                    loss = criterion(logit, sag_labels) / loss_denom
                    loss.backward()
                    if is_step:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        optimizer.step()
                        optimizer.zero_grad()
                rl += loss.item() * loss_denom
            else:
                with torch.no_grad():
                    if scaler:
                        with torch.cuda.amp.autocast():
                            logit = model(vols)
                            loss = criterion(logit, sag_labels)
                    else:
                        logit = model(vols)
                        loss = criterion(logit, sag_labels)
                rl += loss.item()

            all_logits.append(logit.detach().float().cpu())
            all_labels_list.append(labels[:, 0].cpu())

    all_logits = torch.cat(all_logits)
    all_labels_t = torch.cat(all_labels_list)

    m = compute_detailed_metrics(all_logits, all_labels_t)
    m["loss"] = rl / len(loader)

    probs_np = torch.sigmoid(all_logits.squeeze(1)).numpy()
    labels_np = all_labels_t.numpy()
    if len(set(labels_np.tolist())) > 1:
        m["auc"] = float(roc_auc_score(labels_np, probs_np))
    else:
        m["auc"] = float("nan")

    return m


def _epoch_score(val_metrics):
    if not np.isnan(val_metrics["auc"]):
        return val_metrics["auc"]
    return val_metrics["acc"]


best_val_auc = float("-inf")
best_val_acc_at_best_auc = 0.0
no_imp = 0
history = []
best_path = exp_dir / "best_model.pth"

print(
    f"\n{'Ep':>4} {'Tr-loss':>8} {'Tr-acc':>7} {'Tr-sens':>8} {'Tr-spec':>8} │ "
    f"{'Va-loss':>8} {'Va-acc':>7} {'Va-sens':>8} {'Va-spec':>8} {'Va-F1':>7} {'Va-AUC':>7}"
)
print("─" * 95)

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(True, epoch)
    val = run_epoch(False, epoch)
    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]

    sc = _epoch_score(val)
    print(
        f"{epoch:>4} {tr['loss']:>8.4f} {tr['acc']:>7.3f} {tr['sensitivity']:>8.3f} {tr['specificity']:>8.3f} │ "
        f"{val['loss']:>8.4f} {val['acc']:>7.3f} {val['sensitivity']:>8.3f} {val['specificity']:>8.3f} "
        f"{val['f1']:>7.3f} {val['auc']:>7.3f}  lr={lr_now:.1e}"
    )

    row = {"epoch": epoch, "lr": lr_now}
    row.update({f"train_{k}": v for k, v in tr.items()})
    row.update({f"val_{k}": v for k, v in val.items()})
    history.append(row)
    with open(exp_dir / "metrics.jsonl", "a") as f:
        f.write(_json2.dumps(row) + "\n")

    if sc > best_val_auc:
        best_val_auc = sc
        best_val_acc_at_best_auc = val["acc"]
        no_imp = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "best_val_auc": float(val["auc"]) if not np.isnan(val["auc"]) else None,
                "best_val_score": float(sc),
                "best_val_accuracy": val["acc"],
                "val_metrics": val,
            },
            best_path,
        )
        print(
            f"     ✓ best по val_auc (score={sc:.4f} acc={val['acc']:.3f} sens={val['sensitivity']:.3f} "
            f"spec={val['specificity']:.3f})"
        )
    else:
        no_imp += 1
        if EARLY_STOPPING > 0 and no_imp >= EARLY_STOPPING:
            print(f"     Early stop (нет роста val_auc) на эпохе {epoch}")
            break

best_val_acc = best_val_acc_at_best_auc
print(f"\nГотово. Лучший score (AUC или acc если AUC=nan): {best_val_auc:.4f}  acc на лучшей эпохе: {best_val_acc:.3f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if not history:
    print("Нет history — сначала выполните ячейку с циклом обучения.")
else:
    ep = [h["epoch"] for h in history]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Sagittal binary — кривые обучения", fontsize=14, fontweight="bold")

    axes[0, 0].plot(ep, [h["train_loss"] for h in history], label="Train")
    axes[0, 0].plot(ep, [h["val_loss"] for h in history], label="Val")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(ep, [h["train_acc"] for h in history], label="Train")
    axes[0, 1].plot(ep, [h["val_acc"] for h in history], label="Val")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    auc_y = np.array([h["val_auc"] for h in history], dtype=float)
    auc_ok = ~np.isnan(auc_y)
    axes[0, 2].plot(np.array(ep)[auc_ok], auc_y[auc_ok], color="C2", label="Val AUC")
    axes[0, 2].set_title("Val AUC-ROC")
    axes[0, 2].set_xlabel("Epoch")
    axes[0, 2].legend()
    axes[0, 2].grid(alpha=0.3)

    axes[1, 0].plot(ep, [h["train_sensitivity"] for h in history], label="Train")
    axes[1, 0].plot(ep, [h["val_sensitivity"] for h in history], label="Val")
    axes[1, 0].set_title("Sensitivity (recall non-central)")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(ep, [h["train_specificity"] for h in history], label="Train")
    axes[1, 1].plot(ep, [h["val_specificity"] for h in history], label="Val")
    axes[1, 1].set_title("Specificity (recall central)")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    ax_f1 = axes[1, 2]
    ax_lr = ax_f1.twinx()
    ax_f1.plot(ep, [h["val_f1"] for h in history], color="purple", label="Val F1")
    ax_f1.set_ylabel("F1", color="purple")
    ax_f1.tick_params(axis="y", labelcolor="purple")
    ax_lr.plot(ep, [h["lr"] for h in history], color="gray", linestyle="--", alpha=0.7, label="LR")
    ax_lr.set_ylabel("Learning rate", color="gray")
    ax_lr.tick_params(axis="y", labelcolor="gray")
    ax_f1.set_title("Val F1 и learning rate")
    ax_f1.set_xlabel("Epoch")
    ax_f1.grid(alpha=0.3)

    plt.tight_layout()
    out = exp_dir / "learning_curves.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Сохранено: {out}")


## 7. Threshold Calibration

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report

ckpt = torch.load(best_path, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
_auc_b = ckpt.get("best_val_auc")
_auc_s = f"{_auc_b:.3f}" if _auc_b is not None and not (isinstance(_auc_b, float) and (_auc_b != _auc_b)) else "n/a"
print(f"Best epoch: {ckpt['epoch']}, val AUC (если есть): {_auc_s}, val acc: {ckpt['best_val_accuracy']:.3f}")
print(f"Val metrics at best: {ckpt['val_metrics']}")

all_probs, all_labels_cal = [], []
with torch.no_grad():
    for vols, labels in val_loader:
        logit = model(vols.to(device))
        all_probs.extend(torch.sigmoid(logit.squeeze(1)).cpu().tolist())
        all_labels_cal.extend(labels[:, 0].tolist())

probs_np  = np.array(all_probs)
labels_np = np.array(all_labels_cal)

print(f"\nVal distribution: 0(central)={sum(labels_np==0)}, 1(non-central)={sum(labels_np==1)}")
print(f"Predictions >0.5: {sum(probs_np>0.5)}")

fpr, tpr, thresholds = roc_curve(labels_np, probs_np)
auc = roc_auc_score(labels_np, probs_np)
j_scores = tpr - fpr
best_idx  = int(np.argmax(j_scores))
best_thresh = float(thresholds[best_idx])
acc_at_thresh = float(np.mean((probs_np >= best_thresh) == labels_np))

print(f"\nAUC-ROC: {auc:.4f}")
print(f"Youden's J best threshold: {best_thresh:.4f}")
print(f"Accuracy at threshold: {acc_at_thresh:.4f}")

preds_opt = (probs_np >= best_thresh).astype(int)
print(f"\nConfusion matrix (threshold={best_thresh:.3f}):")
print(confusion_matrix(labels_np, preds_opt, labels=[0,1]))
print(classification_report(labels_np, preds_opt, labels=[0,1], target_names=["central","non-central"]))

calibration = {"optimal_threshold": best_thresh, "auc_roc": auc, "accuracy_at_threshold": acc_at_thresh}
with open(exp_dir/"config.json") as f: cfg = _json2.load(f)
cfg.update(calibration); cfg["best_val_accuracy"] = best_val_acc
with open(exp_dir/"config.json","w") as f: _json2.dump(cfg,f,indent=2)
print(f"Порог сохранён: {best_thresh:.4f}")

### 7.1 Примеры + предсказания модели

Нужны: лучший чекпоинт и `best_thresh` из ячейки выше. Трипланарные срезы тех же кропов; для **sagittal** показаны вероятность класса «нецентр.», предсказание и разметка.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch


def _triplanar_pred(vol_np, axes3, cmap="gray"):
    D, H, W = vol_np.shape
    axes3[0].imshow(vol_np[D // 2, :, :], cmap=cmap, vmin=0, vmax=1)
    axes3[0].set_title("mid-Z")
    axes3[0].axis("off")
    axes3[1].imshow(vol_np[:, H // 2, :], cmap=cmap, vmin=0, vmax=1)
    axes3[1].set_title("mid-Y")
    axes3[1].axis("off")
    axes3[2].imshow(vol_np[:, :, W // 2], cmap=cmap, vmin=0, vmax=1)
    axes3[2].set_title("mid-X")
    axes3[2].axis("off")


@torch.no_grad()
def _sag_probs(ds, indices):
    model.eval()
    out = []
    for i in indices:
        x = torch.from_numpy(ds.cache[int(i)]).float().unsqueeze(0).unsqueeze(0).to(device)
        logit = model(x)
        out.append(float(torch.sigmoid(logit.squeeze()).cpu()))
    return out


def _plot_pred_grid(ds, indices, probs, title, save_path=None):
    def _lab_sag_name(y):
        return "центральное" if int(y) == 0 else "нецентральное"

    n = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(10, 2.7 * n))
    if n == 1:
        axes = np.asarray([axes])
    fig.suptitle(title, fontsize=12, fontweight="bold")
    thresh = float(best_thresh)
    for r, (idx, p) in enumerate(zip(indices, probs)):
        idx = int(idx)
        vol = ds.cache[idx]
        rec = ds.records[idx]
        true_y = int(rec["sag"])
        pred_y = int(p >= thresh)
        ok = pred_y == true_y
        _triplanar_pred(vol, axes[r])

        axes[r, 0].set_ylabel(
            f"{rec['side']}\n{rec['study_id'][:12]}",
            fontsize=8,
            rotation=0,
            labelpad=34,
            va="center",
        )
        col = "#1d3557" if ok else "#e63946"
        axes[r, 1].set_title(
            f"P(класс 1, нецентр.)={p:.3f}\n"
            f"В датасете (sagittal): класс {true_y} — {_lab_sag_name(true_y)}\n"
            f"Предсказано (sagittal): класс {pred_y} — {_lab_sag_name(pred_y)}",
            fontsize=9,
            color=col,
        )
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Сохранено: {save_path}")
    plt.show()


_rng = np.random.default_rng(17)
K_TR = min(3, len(train_ds))
K_VA = min(6, len(val_ds))
tr_ix2 = _rng.choice(len(train_ds), size=K_TR, replace=False)
va_ix2 = _rng.choice(len(val_ds), size=K_VA, replace=False)

p_tr = _sag_probs(train_ds, tr_ix2)
p_va = _sag_probs(val_ds, va_ix2)

_plot_pred_grid(
    train_ds,
    tr_ix2,
    p_tr,
    f"Train: сравнение классов  ·  в датасете vs предсказано (sagittal)  ·  порог Youden J = {float(best_thresh):.3f} (класс 1 если P≥порог)",
    save_path=exp_dir / "viz_train_predictions.png",
)
_plot_pred_grid(
    val_ds,
    va_ix2,
    p_va,
    f"Validation: сравнение классов  ·  в датасете vs предсказано (sagittal)  ·  порог Youden J = {float(best_thresh):.3f}",
    save_path=exp_dir / "viz_val_predictions.png",
)


### 7.2 Ошибки на валидации: FP и FN

**Ложноположительные (FP):** в датасете класс **0** (центральное), модель дала класс **1**.  
**Ложноотрицательные (FN):** в датасете класс **1** (нецентральное), модель дала класс **0**.

Используются те же `labels_np`, `preds_opt`, `probs_np` и порог `best_thresh`, что и в калибровке (порядок совпадает с `val_ds`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Из ячейки калибровки: labels_np, preds_opt, probs_np, best_thresh; val_ds — из §4

def _triplanar_fpfn(vol_np, axes3, cmap="gray"):
    D, H, W = vol_np.shape
    axes3[0].imshow(vol_np[D // 2, :, :], cmap=cmap, vmin=0, vmax=1)
    axes3[0].set_title("mid-Z")
    axes3[0].axis("off")
    axes3[1].imshow(vol_np[:, H // 2, :], cmap=cmap, vmin=0, vmax=1)
    axes3[1].set_title("mid-Y")
    axes3[1].axis("off")
    axes3[2].imshow(vol_np[:, :, W // 2], cmap=cmap, vmin=0, vmax=1)
    axes3[2].set_title("mid-X")
    axes3[2].axis("off")


def _cls_name(y):
    return "центральное" if int(y) == 0 else "нецентральное"


fp_idx = np.where((labels_np == 0) & (preds_opt == 1))[0]
fn_idx = np.where((labels_np == 1) & (preds_opt == 0))[0]
print(
    f"Ложноположительные (FP): {len(fp_idx)}  ·  "
    f"Ложноотрицательные (FN): {len(fn_idx)}  ·  "
    f"порог={float(best_thresh):.4f}",
)

MAX_SHOW = 8


def _plot_error_block(indices, title_ru, explanation, save_path=None):
    n_tot = len(indices)
    if n_tot == 0:
        print(f"Нет примеров ({title_ru}).")
        return
    take = indices[:MAX_SHOW]
    n = len(take)
    fig, axes = plt.subplots(n, 3, figsize=(10, 2.75 * n))
    if n == 1:
        axes = np.asarray([axes])
    extra = f" (показано {n} из {n_tot})" if n_tot > n else ""
    fig.suptitle(f"{title_ru}{extra}\n{explanation}", fontsize=11, fontweight="bold")
    for r, i in enumerate(take):
        i = int(i)
        vol = val_ds.cache[i]
        rec = val_ds.records[i]
        p = float(probs_np[i])
        ty, py = int(labels_np[i]), int(preds_opt[i])
        _triplanar_fpfn(vol, axes[r])
        axes[r, 0].set_ylabel(
            f"{rec['side']}\n{rec['study_id'][:12]}",
            fontsize=8,
            rotation=0,
            labelpad=34,
            va="center",
        )
        axes[r, 1].set_title(
            f"P(класс 1)={p:.3f}\n"
            f"В датасете: класс {ty} — {_cls_name(ty)}\n"
            f"Предсказано: класс {py} — {_cls_name(py)}",
            fontsize=9,
            color="#9d0208",
        )
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Сохранено: {save_path}")
    plt.show()


_plot_error_block(
    fp_idx,
    "Ложноположительные (false positive)",
    "В датасете класс 0 (центральное), модель предсказала класс 1 (нецентральное).",
    save_path=exp_dir / "viz_fp_val.png",
)
_plot_error_block(
    fn_idx,
    "Ложноотрицательные (false negative)",
    "В датасете класс 1 (нецентральное), модель предсказала класс 0 (центральное).",
    save_path=exp_dir / "viz_fn_val.png",
)


## 8. Visualization (ROC)

После раздела 7: ROC и точка порога по Youden J (`roc_curve.png`).

In [ ]:
import matplotlib.pyplot as plt

# Кривые по эпохам — в ячейке сразу после обучения (learning_curves.png)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, lw=2, label=f"AUC={auc:.3f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.scatter(
    [fpr[best_idx]],
    [tpr[best_idx]],
    color="red",
    s=100,
    zorder=5,
    label=f"Youden J, thresh={best_thresh:.3f}",
)
ax.set_xlabel("FPR (1 − specificity)")
ax.set_ylabel("TPR (sensitivity)")
ax.set_title("ROC — Sagittal (central vs non-central)")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(exp_dir / "roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Сохранено: {exp_dir}/roc_curve.png")

## 9. Export

Сводка в JSON, **manifest**, краткий **README**, превью датасета и **ZIP** со всей папкой эксперимента (`experiments/<имя>_bundle.zip`) для скачивания одним файлом.

In [ ]:
import shutil
import numpy as np

analysis = {
    "config": cfg,
    "total_epochs": len(history),
    "best_epoch": ckpt["epoch"],
    "best_val_accuracy": best_val_acc,
    "calibration": calibration,
    "history": history,
}
ap = exp_dir / "training_analysis.json"
with open(ap, "w") as f: _json2.dump(analysis, f, indent=2, ensure_ascii=False)
print(f"Analysis: {ap}")
print(f"Model:    {best_path}")
print(f"\nИтог — Best val acc: {best_val_acc:.3f}")
print(
    f"Калибровка: AUC={calibration.get('auc_roc')}, "
    f"порог={calibration.get('optimal_threshold')}, "
    f"accuracy@threshold={calibration.get('accuracy_at_threshold')}",
)

readme = exp_dir / "README_СКАЧАТЬ_ЭКСПЕРИМЕНТ.md"
readme.write_text(
    """# Артефакты прогона обучения

Все файлы лежат в **этой папке** (`sag_only_*`). Можно скачать каталог целиком или готовый **ZIP** (`…_bundle.zip`) из той же родительской директории (`experiments/`).

| Файл | Назначение |
|------|------------|
| `config.json` | Гиперпараметры и калибровка порога |
| `metrics.jsonl` | Метрики по эпохам |
| `best_model.pth` | Лучший чекпоинт по val AUC (иначе по val accuracy) |
| `training_analysis.json` | Сводка: config + history + calibration |
| `learning_curves.png` | Кривые обучения |
| `roc_curve.png` | ROC и порог Youden J |
| `viz_train_predictions.png`, `viz_val_predictions.png` | Примеры предсказаний |
| `viz_fp_val.png`, `viz_fn_val.png` | FP / FN на валидации (если были ошибки) |
| `dataset_preview.png` | Случайные кропы train/val (создаётся при экспорте) |
| `manifest.json` | Список файлов и размеры |

Классы **sagittal:** 0 = центральное, 1 = нецентральное.

**DataSphere:** `filestore/experiments/`. **Colab:** `/content/experiments/`.
""",
    encoding="utf-8",
)

try:
    import matplotlib.pyplot as plt

    if "train_ds" in globals() and len(train_ds) > 0 and len(val_ds) > 0:
        rng = np.random.default_rng(42)
        n_tr = min(3, len(train_ds))
        n_va = min(3, len(val_ds))
        tr_ix = rng.choice(len(train_ds), size=n_tr, replace=False)
        va_ix = rng.choice(len(val_ds), size=n_va, replace=False)

        def _tp(vol, ax3):
            d, h, w = vol.shape
            ax3[0].imshow(vol[d // 2, :, :], cmap="gray", vmin=0, vmax=1)
            ax3[1].imshow(vol[:, h // 2, :], cmap="gray", vmin=0, vmax=1)
            ax3[2].imshow(vol[:, :, w // 2], cmap="gray", vmin=0, vmax=1)
            for a in ax3:
                a.axis("off")

        nrows = n_tr + n_va
        fig, axes = plt.subplots(nrows, 3, figsize=(9, 2.2 * nrows))
        if nrows == 1:
            axes = np.asarray([axes])
        r = 0
        for ix in tr_ix:
            ix = int(ix)
            _tp(train_ds.cache[ix], axes[r])
            axes[r, 0].set_ylabel("train", fontsize=8)
            r += 1
        for ix in va_ix:
            ix = int(ix)
            _tp(val_ds.cache[ix], axes[r])
            axes[r, 0].set_ylabel("val", fontsize=8)
            r += 1
        fig.suptitle("Dataset preview (export)", fontsize=10)
        plt.tight_layout()
        dp = exp_dir / "dataset_preview.png"
        fig.savefig(dp, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"Сохранено: {dp}")
except Exception as _e:
    print("dataset_preview:", _e)

manifest = {"experiment_dir": str(exp_dir.resolve()), "files": []}
for p in sorted(exp_dir.iterdir()):
    if p.is_file():
        manifest["files"].append({"name": p.name, "bytes": p.stat().st_size})
with open(exp_dir / "manifest.json", "w") as f:
    _json2.dump(manifest, f, indent=2, ensure_ascii=False)

bundle_base = exp_dir.parent / f"{exp_dir.name}_bundle"
shutil.make_archive(str(bundle_base), "zip", root_dir=str(exp_dir.parent), base_dir=exp_dir.name)
print(f"\nZIP (вся папка эксперимента):\n  {bundle_base}.zip")
print(f"Каталог для скачивания:\n  {exp_dir.resolve()}")